In [599]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from xgboost import XGBClassifier
import joblib

print("Loading master features CSV...")
df = pd.read_csv(Path.cwd().parent /'master_dataset_lexical_features.csv')

Loading master features CSV...


In [600]:
X = df.drop(['url', 'label'], axis=1)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Data split successful. Training shape: {X_train.shape}, Testing shape: {X_test.shape}")

Data split successful. Training shape: (708018, 20), Testing shape: (177005, 20)


In [601]:
print("Initializing XGBoost Classifier...")

model = XGBClassifier(
    n_estimators=2000,
    learning_rate=0.08,
    max_depth=8,          # ← up from 6
    objective='binary:logistic',
    random_state=42,
    n_jobs=-1,
    colsample_bytree=0.9, # ← slightly more features per tree
    colsample_bylevel=0.9,
    subsample=0.85,
    #scale_pos_weight =2,
    reg_alpha=0.1,
    reg_lambda = 1.5,
    min_child_weight=1,   # ← down from 5, allows finer splits
)

print("Training the sequential gradient boosting pipeline...")
model.fit(X_train, y_train)
print("XGBoost training cycle complete!")

Initializing XGBoost Classifier...
Training the sequential gradient boosting pipeline...
XGBoost training cycle complete!


In [602]:
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)  # default is 0.5, lower = catch more malicious

accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=['Safe', 'Malicious'])

print("\n================ XGBOOST METRICS ================")
print(f"Target Threshold Clear State: {accuracy * 100:.2f}%")
print("-------------------------------------------------")
print("Confusion Matrix Layout:")
print(cm)
print("-------------------------------------------------")
print("Classification Matrix Specs:")
print(report)


================ XGBOOST METRICS ================
Target Threshold Clear State: 94.75%
-------------------------------------------------
Confusion Matrix Layout:
[[116324   3676]
 [  5623  51382]]
-------------------------------------------------
Classification Matrix Specs:
              precision    recall  f1-score   support

        Safe       0.95      0.97      0.96    120000
   Malicious       0.93      0.90      0.92     57005

    accuracy                           0.95    177005
   macro avg       0.94      0.94      0.94    177005
weighted avg       0.95      0.95      0.95    177005



In [603]:
xgb_model = Path.cwd().parent /'backend' /'ml_model_xgb.joblib'

joblib.dump(model, xgb_model)

['/Users/AmeyaWalekar/Desktop/Summer 2026/Phisguard - CC Project/The Project /phishguard/backend/ml_model_xgb.joblib']

In [604]:
#testing

feat_imp = pd.Series(model.feature_importances_, index=X_train.columns)
print(feat_imp.sort_values(ascending=False))

path_depth                0.289581
dot_count                 0.176146
is_ip                     0.156745
subdomain_count           0.064971
tld_length                0.052949
brand_in_subdomain        0.036608
is_url_shortener          0.031021
url_length                0.030665
special_char_count        0.029379
suspicious_tld            0.020257
numeric_token_count       0.019833
hostname_length           0.018616
hyphen_count              0.014359
digit_ratio               0.012170
url_digit_ratio           0.011564
longest_hostname_token    0.010667
has_port                  0.009083
at_count                  0.008121
url_entropy               0.007267
query_count               0.000000
dtype: float32


In [605]:
import joblib

model = joblib.load(Path.cwd().parent /'backend'/"ml_model_xgb.joblib")
print("Expected features:", getattr(model, "n_features_in_", "not available"))
print("Feature names:", getattr(model, "feature_names_in_", "not available"))

Expected features: 20
Feature names: ['url_length' 'hostname_length' 'dot_count' 'hyphen_count' 'at_count'
 'query_count' 'is_ip' 'url_entropy' 'subdomain_count' 'suspicious_tld'
 'digit_ratio' 'has_port' 'path_depth' 'brand_in_subdomain'
 'is_url_shortener' 'url_digit_ratio' 'special_char_count'
 'longest_hostname_token' 'numeric_token_count' 'tld_length']


In [606]:
# testing

import pandas as pd
feat_imp = pd.Series(model.feature_importances_, index=X_train.columns)
print(feat_imp.sort_values(ascending=False))

path_depth                0.289581
dot_count                 0.176146
is_ip                     0.156745
subdomain_count           0.064971
tld_length                0.052949
brand_in_subdomain        0.036608
is_url_shortener          0.031021
url_length                0.030665
special_char_count        0.029379
suspicious_tld            0.020257
numeric_token_count       0.019833
hostname_length           0.018616
hyphen_count              0.014359
digit_ratio               0.012170
url_digit_ratio           0.011564
longest_hostname_token    0.010667
has_port                  0.009083
at_count                  0.008121
url_entropy               0.007267
query_count               0.000000
dtype: float32
